# 02 — Lag Analysis

**The question:** how many weeks after a rainfall anomaly do malaria cases rise?

**Why it is not a fixed number:** published rainfall-to-malaria lags range from
2–8 weeks in some ecologies to 1–3 months in others, depending on vector species,
altitude, and whether breeding sites are rain-fed pools or permanent water. A
coefficient fitted in the Kenyan highlands is simply wrong on the Dar es Salaam
coast (shortcoming #8).

This notebook demonstrates the platform's answer: **the YAML config supplies a
search range and a prior; the system fits the actual lag per district**
(critical rule #3). If every district agreed, per-district fitting would be
wasted effort — so the key output here is the *dispersion* of fitted lags.

In [ ]:
# Make the repo importable regardless of where Jupyter was launched from.
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "config").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import logging
logging.getLogger("afya").setLevel(logging.WARNING)   # keep notebook output readable

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print(f"repo root: {ROOT}")

In [ ]:
# Plotting is optional throughout these notebooks: matplotlib is not a hard
# dependency of AFYA-PREDICT, because the platform must install on low-spec
# district hardware. Every notebook falls back to printed tables without it.
#
# Backend selection matters more than it looks. Inside a Jupyter kernel,
# matplotlib configures its own inline backend and we leave it alone. Anywhere
# else - `nbconvert --execute`, CI, a headless server - a GUI backend will block
# forever on a window that never opens (a set-but-unreachable $DISPLAY is enough
# to trigger it), so we force the non-interactive Agg backend.
import os
import sys

try:
    import matplotlib
    _in_kernel = "ipykernel" in sys.modules
    if not os.environ.get("MPLBACKEND") and not _in_kernel:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (11, 4)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    HAS_PLT = True
    print(f"matplotlib {matplotlib.__version__} on the "
          f"{matplotlib.get_backend()} backend")
except ImportError:
    HAS_PLT = False
    print("matplotlib not installed - tables will be printed instead of plotted")

In [ ]:
from src.core.config_loader import load_region_config, load_disease_config
from src.core.geo import subset_region

FULL_REGION = load_region_config("tanzania")
print(f"{len(FULL_REGION.districts)} councils across "
      f"{len({d.region for d in FULL_REGION.districts})} regions")

# A small, ecologically diverse subset keeps these notebooks fast to run.
# Swap in FULL_REGION for a national analysis (much slower).
STUDY_DISTRICTS = [
    "Kinondoni",     # dense coastal city
    "Ilala",         # dense coastal city, adjacent to Kinondoni
    "Mwanza City",   # lakeside city
    "Sengerema",     # rural lakeside, low WASH coverage
    "Dodoma City",   # semi-arid central
    "Songea MC",     # southern highlands
]
REGION = subset_region(FULL_REGION, STUDY_DISTRICTS)
pd.DataFrame([d.model_dump() for d in REGION.districts]).set_index("name")

## 1. What the config declares

Each digital proxy carries a lag *search range*, a *prior*, and — importantly —
the causal *mechanism* it represents. The mechanism is what lets an explanation
say "rainfall six weeks ago floods breeding sites" instead of "feature 37 = 0.42".

In [ ]:
malaria = load_disease_config("malaria")
cholera = load_disease_config("cholera")

def proxy_table(config):
    return pd.DataFrame([{
        "proxy": p.name,
        "source": p.source,
        "relationship": p.relationship.value,
        "search_range_weeks": f"{p.lag_weeks_range[0]}-{p.lag_weeks_range[1]}",
        "prior_lag": p.optimal_lag_weeks,
        "mechanism": p.mechanism,
    } for p in config.digital_proxies])

print(f"=== {malaria.name} ({malaria.code}) ===")
display(proxy_table(malaria))
print(f"\n=== {cholera.name} ({cholera.code}) ===")
display(proxy_table(cholera))

Note the two very different lags cholera declares for the same climate system:
`rainfall` at 1–8 weeks (flooding contaminates water within days) and
`temperature` at 8–20 weeks (reservoir warming acts on a ~4-month lag). A model
forced to pick one would lose the other.

## 2. Ingest history

Lag fitting needs several seasons; two years is the practical floor.

In [ ]:
from src.data_ingestion.normalizer import ingest

SOURCES = sorted(set(malaria.required_sources) | set(cholera.required_sources) | {"dhis2"})
panel = ingest(SOURCES, "2019-W01", "2024-W52", region=REGION)
values = panel.values()
print(f"{len(panel.weeks)} weeks x {len(panel.districts)} districts, "
      f"{len(panel.value_columns)} variables")

## 3. The lag scan, done by hand first

Before using the platform's fitter, here is exactly what it does: shift the
driver by each candidate lag, correlate against the target, keep the peak.

Spearman rather than Pearson, deliberately — several proxy relationships are
monotone but *not linear* (rainfall saturates, temperature is bell-shaped), and
rank correlation is robust to the reporting spikes surveillance data carries.

In [ ]:
def lag_curve(district, driver, target, max_lag=20):
    local = values.xs(district, level="district").sort_index()
    if driver not in local or target not in local:
        return pd.Series(dtype=float)
    rows = {}
    for lag in range(max_lag + 1):
        pair = pd.concat([local[driver].shift(lag), local[target]], axis=1).dropna()
        rows[lag] = (pair.iloc[:, 0].corr(pair.iloc[:, 1], method="spearman")
                     if len(pair) > 20 else np.nan)
    return pd.Series(rows, name=district)

curves = pd.DataFrame({d: lag_curve(d, "rainfall_mm", "cases_malaria")
                       for d in REGION.district_names})

print("Peak-correlation lag, rainfall -> malaria, per district:")
peaks = pd.DataFrame({
    "peak_lag_weeks": curves.abs().idxmax(),
    "rho_at_peak": [curves[d].loc[curves[d].abs().idxmax()] for d in curves.columns],
}).round(3)
display(peaks)
print(f"\nSpread: {int(peaks.peak_lag_weeks.min())}-{int(peaks.peak_lag_weeks.max())} weeks "
      f"across just {len(peaks)} districts. The malaria config's prior is "
      f"{next(p.optimal_lag_weeks for p in malaria.digital_proxies if p.name == 'rainfall')} weeks.")

if HAS_PLT:
    ax = curves.plot(figsize=(11, 4.5), marker="o", ms=3)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("rainfall lead time (weeks)"); ax.set_ylabel("Spearman rho")
    ax.set_title("Rainfall -> malaria cross-correlation, per district")
    ax.legend(fontsize=8, ncol=3)
    plt.show()

## 4. The platform's fitter

`fit_optimal_lags` runs that scan for every proxy in every district, and — this
is the part that matters for sparse councils — falls back gracefully:

| Districts with | Strategy | Recorded as |
|---|---|---|
| ≥60 usable observations | fit its own lag | `fitted` |
| too few observations | borrow the pooled national fit | `pooled` |
| a single-value lag range | use the config prior | `prior` |

A district never gets a fabricated local fit. What it got is always visible in
the `source` column, so a forecast built on borrowed structure is auditable.

In [ ]:
from src.feature_engineering.builder import PROXY_TO_VARIABLE
from src.feature_engineering.lag_features import (
    fit_optimal_lags, lag_dispersion, lag_fit_report,
)

specs = [p for p in malaria.digital_proxies if not p.optional]
variable_for_proxy = {p.name: PROXY_TO_VARIABLE.get(p.name, p.name) for p in specs}

fits = fit_optimal_lags(values, "cases_malaria", specs, variable_for_proxy)
report = lag_fit_report(fits)

print("Fitted lag per district x proxy:\n")
display(report.pivot(index="district", columns="proxy", values="lag_weeks"))
print("\nHow each lag was arrived at:\n")
display(report.groupby(["proxy", "source"]).size().rename("districts").reset_index())

## 5. The headline result: districts disagree

`lag_dispersion` is the evidence for shortcoming #8. If `unique_lags` is 1 for
every proxy, a single national coefficient would have been fine. It is not.

In [ ]:
dispersion = lag_dispersion(fits)
display(dispersion)

priors = {p.name: p.optimal_lag_weeks for p in specs}
comparison = dispersion.assign(
    config_prior=dispersion["proxy"].map(priors),
    spread_weeks=dispersion["max_lag"] - dispersion["min_lag"],
)
comparison["prior_matches_median"] = comparison["config_prior"] == comparison["median_lag"]

print("\nFitted lags versus the config's prior:\n")
display(comparison[["proxy", "config_prior", "min_lag", "median_lag", "max_lag",
                    "spread_weeks", "unique_lags", "prior_matches_median"]])

disagreeing = int((comparison["unique_lags"] > 1).sum())
print(f"\n{disagreeing} of {len(comparison)} proxies show districts disagreeing on the lag.")
print("Each week of disagreement is a week of forecast error a transplanted")
print("coefficient would have introduced.")

## 6. Does it recover a lag we planted?

Correlation peaks can be coincidence. The honest test is a controlled one:
construct a series where the true lag is known, and check the fitter finds it
*despite* being given a misleading prior.

In [ ]:
from src.core.types import FeatureSpec

weeks = [f"{y}-W{w:02d}" for y in (2021, 2022, 2023) for w in range(1, 53)]
rng = np.random.default_rng(7)

synthetic_rows = []
TRUE_LAGS = {"PlantedA": 5, "PlantedB": 11}
for district, true_lag in TRUE_LAGS.items():
    rain = 45 + 35 * np.sin(np.linspace(0, 6 * np.pi, len(weeks))) + rng.normal(0, 5, len(weeks))
    cases = np.roll(rain, true_lag) * 3.5 + rng.normal(0, 9, len(weeks))
    for i, week in enumerate(weeks):
        synthetic_rows.append({"district": district, "week": week,
                               "rainfall_mm": rain[i], "cases_malaria": max(cases[i], 0)})

planted = pd.DataFrame(synthetic_rows).set_index(["district", "week"]).sort_index()

MISLEADING_PRIOR = 2
spec = FeatureSpec(name="rainfall", source="chirps", lag_weeks_range=(1, 16),
                   optimal_lag_weeks=MISLEADING_PRIOR, mechanism="controlled test")
planted_fits = fit_optimal_lags(planted, "cases_malaria", [spec], {"rainfall": "rainfall_mm"})

print(f"Config prior given to the fitter: {MISLEADING_PRIOR} weeks (deliberately wrong)\n")
for district, true_lag in TRUE_LAGS.items():
    fit = planted_fits[district]["rainfall"]
    status = "recovered" if abs(fit.lag_weeks - true_lag) <= 1 else "MISSED"
    print(f"  {district}: true {true_lag:>2}w -> fitted {fit.lag_weeks:>2}w "
          f"(rho {fit.score:.3f}, via {fit.source})  [{status}]")

## 7. How the fitted lag becomes a feature

The fitter's answer is not used as a single hard-coded column. The builder
materialises the fitted lag *and its immediate neighbours* (±1 week, clipped to
the declared range), so the model can refine the choice the rank-correlation
scan made — correlation finds the neighbourhood, the model finds the point.

In [ ]:
from src.feature_engineering.builder import FeatureBuilder

builder = FeatureBuilder(malaria, REGION)
matrix = builder.build(panel, lag_fits=fits)

lag_features = sorted(f for f in matrix.feature_names if "_lag" in f and "cases_" not in f)
print(f"{len(matrix.feature_names)} features total, {len(lag_features)} lagged driver columns\n")

provenance = pd.DataFrame([
    {**matrix.describe_feature(f)} for f in lag_features
])[["feature", "proxy", "lag_weeks", "source", "relationship"]]
display(provenance.head(15))

print("\nLags materialised per proxy:")
display(provenance.groupby("proxy")["lag_weeks"].apply(lambda s: sorted(s.unique())).rename("lags"))

## 8. Mechanism-shaped features

Fitting the lag is only half of encoding the biology. The YAML also declares the
*shape* of each dose-response, and `apply_response_shape` transforms the raw
driver accordingly — so rainfall's saturation and temperature's optimum are
represented explicitly rather than left for the model to rediscover from scratch.

In [ ]:
from src.feature_engineering.interaction_terms import apply_response_shape

rain_spec = next(p for p in malaria.digital_proxies if p.name == "rainfall")
temp_spec = next(p for p in malaria.digital_proxies if p.name == "temperature")

rain_grid = pd.Series(np.linspace(0, 320, 120))
temp_grid = pd.Series(np.linspace(14, 40, 120))
rain_shaped = apply_response_shape(rain_grid, rain_spec)
temp_shaped = apply_response_shape(temp_grid, temp_spec)

print(f"rainfall  : {rain_spec.relationship.value}, {rain_spec.params()}")
print(f"           peaks at {rain_grid[rain_shaped.idxmax()]:.0f} mm, "
      f"then declines - breeding sites wash out")
print(f"temperature: {temp_spec.relationship.value}, {temp_spec.params()}")
print(f"           peaks at {temp_grid[temp_shaped.idxmax()]:.1f} C")

if HAS_PLT:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.4))
    a1.plot(rain_grid, rain_shaped); a1.axvline(150, ls="--", c="r", lw=1)
    a1.set_title("rainfall: positive with saturation"); a1.set_xlabel("mm")
    a2.plot(temp_grid, temp_shaped); a2.axvspan(26, 28, alpha=0.2, color="tab:orange")
    a2.set_title("temperature: bell curve"); a2.set_xlabel("deg C")
    plt.tight_layout(); plt.show()

## Takeaways

1. **Districts genuinely disagree on transmission lags** — section 5 quantifies it.
   This is the empirical case for per-district fitting.
2. **The fitter recovers a planted lag despite a wrong prior** (section 6), so the
   config's estimates are a starting point, not an assumption baked into results.
3. **Sparse districts borrow rather than fabricate**, and the borrowing is recorded.
4. **Mechanism is carried through** from YAML to feature to explanation, which is
   what makes the SHAP output in notebook 03 readable by an epidemiologist.

Next: **`03_model_training.ipynb`** trains on these features and explains a forecast.